In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ============================================================
# PATHS
# ============================================================
VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'
AREAS_PATH  = '/home/maria/ProjectionSort/data/brain_area.npy'

# ============================================================
# LOAD DATA
# ============================================================
vit   = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R     = np.load(NEURAL_PATH).T                                   # (images, neurons)
areas = np.load(AREAS_PATH, allow_pickle=True)

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# ============================================================
# ANIMATE / INANIMATE LABEL
# ============================================================
top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)
print("Animate fraction:", y.mean())

# ============================================================
# STANDARDIZE NEURAL FEATURES
# ============================================================
scaler = StandardScaler(with_mean=True, with_std=True)
X = scaler.fit_transform(R)   # (n_images, n_neurons)

# ============================================================
# HELPERS: FISHER SUMMARIES WITHOUT NxN
# ============================================================
def fit_logreg_and_fisher_summaries(X, y, C=1.0, max_iter=2000, seed=0):
    """
    Fit logistic regression and compute cheap Fisher summaries.

    Returns:
      beta: (n_features,)
      p:    (n_samples,) predicted probabilities for class 1
      w:    (n_samples,) p*(1-p)
      S_axis: scalar beta^T F beta = sum_i w_i (x_i^T beta)^2   (Wald-energy along learned axis)
      trace_diag: scalar tr(diag(F)) = sum_j sum_i w_i x_ij^2   (overall diagonal curvature)
      fisher_diag: (n_features,) diag(F) = sum_i w_i x_ij^2
    """
    # Use L2 penalty; solver supports high-d
    clf = LogisticRegression(
        penalty="l2",
        C=C,
        solver="lbfgs",
        max_iter=max_iter,
        random_state=seed,
        n_jobs=None,
    )
    clf.fit(X, y)

    beta = clf.coef_.ravel()           # (n_features,)
    p = clf.predict_proba(X)[:, 1]     # (n_samples,)
    w = p * (1.0 - p)                  # (n_samples,)

    # Axis energy: beta^T X^T W X beta = sum_i w_i (x_i^T beta)^2
    z = X @ beta                       # (n_samples,)
    S_axis = float(np.sum(w * (z ** 2)))

    # Diagonal Fisher: diag(F)_j = sum_i w_i x_ij^2
    fisher_diag = np.sum((w[:, None] * (X ** 2)), axis=0)        # (n_features,)
    trace_diag = float(np.sum(fisher_diag))

    return beta, p, w, S_axis, trace_diag, fisher_diag

# ============================================================
# TRUE LABELS
# ============================================================
beta_true, p_true, w_true, S_true, tr_true, fdiag_true = fit_logreg_and_fisher_summaries(
    X, y, C=1.0, max_iter=3000, seed=0
)

print("\n=== TRUE LABELS ===")
print("S_axis = beta^T F beta:", S_true)
print("trace_diag(F):", tr_true)
print("fisher_diag median/mean:", np.median(fdiag_true), np.mean(fdiag_true))

# ============================================================
# PERMUTATION NULL (shuffle labels, refit, recompute)
# ============================================================
n_perm = 500          # bump to 2000+ if you want a tighter p-value
rng = np.random.default_rng(0)

S_null = np.empty(n_perm, dtype=float)
tr_null = np.empty(n_perm, dtype=float)

for k in range(n_perm):
    y_perm = rng.permutation(y)
    _, _, _, S_k, tr_k, _ = fit_logreg_and_fisher_summaries(
        X, y_perm, C=1.0, max_iter=3000, seed=k+1
    )
    S_null[k] = S_k
    tr_null[k] = tr_k

# One-sided p-values: how often null >= observed
p_S  = (1.0 + np.sum(S_null  >= S_true))  / (n_perm + 1.0)
p_tr = (1.0 + np.sum(tr_null >= tr_true)) / (n_perm + 1.0)

print("\n=== PERMUTATION NULL RESULTS ===")
print(f"n_perm = {n_perm}")
print("S_axis null:   mean/median/max =", S_null.mean(), np.median(S_null), S_null.max())
print("trace null:    mean/median/max =", tr_null.mean(), np.median(tr_null), tr_null.max())
print("\n=== SIGNIFICANCE (one-sided) ===")
print("p-value for S_axis (beta^T F beta):", p_S)
print("p-value for trace_diag(F):", p_tr)

# Optional: effect sizes
print("\n=== EFFECT SIZES ===")
print("S_true / null_median:", S_true / np.median(S_null))
print("tr_true / null_median:", tr_true / np.median(tr_null))


Images: 118
Neurons: 39209
Animate fraction: 0.5338983050847458

=== TRUE LABELS ===
S_axis = beta^T F beta: 1.5355582104944652
trace_diag(F): 807.4684314521191
fisher_diag median/mean: 0.021001596729716177 0.02059395627157334

=== PERMUTATION NULL RESULTS ===
n_perm = 500
S_axis null:   mean/median/max = 1.7953205433104578 1.8812200780625354 2.442298378967136
trace null:    mean/median/max = 995.6189677766567 1045.3712682351184 1442.6961797684992

=== SIGNIFICANCE (one-sided) ===
p-value for S_axis (beta^T F beta): 0.7944111776447106
p-value for trace_diag(F): 0.8223552894211577

=== EFFECT SIZES ===
S_true / null_median: 0.8162565498853984
tr_true / null_median: 0.7724226368066857


In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit

# ============================================================
# PATHS
# ============================================================
VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'

# ============================================================
# LOAD DATA
# ============================================================
vit = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R   = np.load(NEURAL_PATH).T                                   # (images, neurons)

top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)
print("Images:", vit.shape[0], "Neurons:", R.shape[1], "Animate frac:", y.mean())

# ============================================================
# TRAIN -> TEST SPLIT (stratified)
# ============================================================
seed = 0
test_size = 0.2
splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
train_idx, test_idx = next(splitter.split(R, y))

Xtr_raw, Xte_raw = R[train_idx], R[test_idx]
ytr, yte = y[train_idx], y[test_idx]

# Fit scaler on TRAIN only (no leakage)
scaler = StandardScaler(with_mean=True, with_std=True)
Xtr = scaler.fit_transform(Xtr_raw)
Xte = scaler.transform(Xte_raw)

print("Train:", Xtr.shape, "Test:", Xte.shape)

# ============================================================
# Fisher summaries on a given dataset, for a FIXED beta
# ============================================================
def fisher_summaries_given_beta(X, beta):
    """
    Expected Fisher for logistic model on dataset X, using fixed beta:
      p = sigmoid(X beta)
      w = p(1-p)
      S_axis = sum_i w_i (x_i^T beta)^2
      trace_diag = sum_j sum_i w_i x_ij^2
    """
    z = X @ beta
    p = 1.0 / (1.0 + np.exp(-z))
    w = p * (1.0 - p)

    S_axis = float(np.sum(w * (z ** 2)))
    fisher_diag = np.sum((w[:, None] * (X ** 2)), axis=0)
    trace_diag = float(np.sum(fisher_diag))

    return S_axis, trace_diag

# ============================================================
# Fit model on TRAIN labels -> evaluate Fisher on TEST
# ============================================================
def fit_beta(Xtr, ytr, C=1.0, seed=0, max_iter=3000):
    clf = LogisticRegression(
        penalty="l2",
        C=C,
        solver="lbfgs",
        max_iter=max_iter,
        random_state=seed
    )
    clf.fit(Xtr, ytr)
    return clf.coef_.ravel()

C = 1.0
beta_true = fit_beta(Xtr, ytr, C=C, seed=seed)

S_true_test, tr_true_test = fisher_summaries_given_beta(Xte, beta_true)

print("\n=== TRUE (trained on true labels) ===")
print("TEST S_axis:", S_true_test)
print("TEST trace_diag:", tr_true_test)

# ============================================================
# PERMUTATION NULL:
# shuffle TRAIN labels, refit beta, score on the SAME TEST SET
# ============================================================
n_perm = 500
rng = np.random.default_rng(0)

S_null_test = np.empty(n_perm, dtype=float)
tr_null_test = np.empty(n_perm, dtype=float)

for k in range(n_perm):
    ytr_perm = rng.permutation(ytr)   # permute only TRAIN labels
    beta_perm = fit_beta(Xtr, ytr_perm, C=C, seed=k+1)
    S_k, tr_k = fisher_summaries_given_beta(Xte, beta_perm)
    S_null_test[k] = S_k
    tr_null_test[k] = tr_k

p_S  = (1.0 + np.sum(S_null_test  >= S_true_test)) / (n_perm + 1.0)
p_tr = (1.0 + np.sum(tr_null_test >= tr_true_test)) / (n_perm + 1.0)

print("\n=== PERMUTATION NULL (test-set Fisher) ===")
print("S_axis null mean/median/max:", S_null_test.mean(), np.median(S_null_test), S_null_test.max())
print("trace null mean/median/max:", tr_null_test.mean(), np.median(tr_null_test), tr_null_test.max())

print("\n=== SIGNIFICANCE (one-sided) ===")
print("p-value S_axis on TEST:", p_S)
print("p-value trace_diag on TEST:", p_tr)

print("\n=== EFFECT SIZES ===")
print("S_true_test / null_median:", S_true_test / np.median(S_null_test))
print("tr_true_test / null_median:", tr_true_test / np.median(tr_null_test))


Images: 118 Neurons: 39209 Animate frac: 0.5338983050847458
Train: (94, 39209) Test: (24, 39209)

=== TRUE (trained on true labels) ===
TEST S_axis: 5.01310886037205
TEST trace_diag: 56215.85661292978

=== PERMUTATION NULL (test-set Fisher) ===
S_axis null mean/median/max: 5.808390950595778 5.786585228961924 7.983833743953161
trace null mean/median/max: 140694.5024047221 140726.5870848937 204052.04657332704

=== SIGNIFICANCE (one-sided) ===
p-value S_axis on TEST: 0.8303393213572854
p-value trace_diag on TEST: 1.0

=== EFFECT SIZES ===
S_true_test / null_median: 0.8663328477875664
tr_true_test / null_median: 0.39946862762341717
